In [1]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt

In [2]:
def read_ops(list_session_data_path):
    list_ops = []
    for session_data_path in list_session_data_path:
        ops = np.load(
            os.path.join(session_data_path, 'ops.npy'),
            allow_pickle=True).item()
        ops['save_path0'] = os.path.join(session_data_path)
        list_ops.append(ops)
    return list_ops

In [3]:
list_session_data_path = [
    'F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250814_2afc-584',
    'F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250811_2afc-580',
    'F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250808_2afc-577',
    'F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250805_2afc-568',
]

list_ops = read_ops(list_session_data_path)

In [4]:
# create a numpy memmap from an h5py dataset.
def create_memmap(data, dtype, mmap_path):
    memmap_arr = np.memmap(mmap_path, dtype=dtype, mode='w+', shape=data.shape)
    memmap_arr[:] = data[...]
    return memmap_arr

# create folder for h5 data.
def get_memmap_path(ops, h5_file_name):
    mm_folder_name, _ = os.path.splitext(h5_file_name)
    if not os.path.exists(os.path.join(ops['save_path0'], 'memmap', mm_folder_name)):
        os.makedirs(os.path.join(ops['save_path0'], 'memmap', mm_folder_name))
    mm_path = os.path.join(ops['save_path0'], 'memmap', mm_folder_name)
    file_path = os.path.join(ops['save_path0'], h5_file_name)
    return mm_path, file_path

####################################################################
# read masks.
def read_masks(ops):
    mm_path, file_path = get_memmap_path(ops, 'masks.h5')
    with h5py.File(file_path, 'r') as f:
        labels     = create_memmap(f['labels'],     'int8',    os.path.join(mm_path, 'labels.mmap'))
        masks      = create_memmap(f['masks_func'], 'float32', os.path.join(mm_path, 'masks_func.mmap'))
        mean_func  = create_memmap(f['mean_func'],  'float32', os.path.join(mm_path, 'mean_func.mmap'))
        max_func   = create_memmap(f['max_func'],   'float32', os.path.join(mm_path, 'max_func.mmap'))
        mean_anat  = create_memmap(f['mean_anat'],  'float32', os.path.join(mm_path, 'mean_anat.mmap')) if ops['nchannels'] == 2 else None
        masks_anat = create_memmap(f['masks_anat'], 'float32', os.path.join(mm_path, 'masks_anat.mmap')) if ops['nchannels'] == 2 else None
    return [labels, masks, mean_func, max_func, mean_anat, masks_anat]

In [5]:
# read dff traces.
def read_dff(ops):
    mm_path, file_path = get_memmap_path(ops, 'dff.h5')
    with h5py.File(file_path, 'r') as f:
        dff = create_memmap(f['dff'], 'float32', os.path.join(mm_path, 'dff.mmap'))
    return dff

In [6]:
# read raw_voltages.h5.
def read_raw_voltages(ops):
    mm_path, file_path = get_memmap_path(ops, 'raw_voltages.h5')
    with h5py.File(file_path, 'r') as f:
        vol_time     = create_memmap(f['raw']['vol_time'],     'float32', os.path.join(mm_path, 'vol_time.mmap'))
        vol_start    = create_memmap(f['raw']['vol_start'],    'int8',    os.path.join(mm_path, 'vol_start.mmap'))
        vol_stim_vis = create_memmap(f['raw']['vol_stim_vis'], 'int8',    os.path.join(mm_path, 'vol_stim_vis.mmap'))
        vol_hifi     = create_memmap(f['raw']['vol_hifi'],     'int8',    os.path.join(mm_path, 'vol_hifi.mmap'))
        vol_img      = create_memmap(f['raw']['vol_img'],      'int8',    os.path.join(mm_path, 'vol_img.mmap'))
        vol_stim_aud = create_memmap(f['raw']['vol_stim_aud'], 'float32', os.path.join(mm_path, 'vol_stim_aud.mmap'))
        vol_flir     = create_memmap(f['raw']['vol_flir'],     'int8',    os.path.join(mm_path, 'vol_flir.mmap'))
        vol_pmt      = create_memmap(f['raw']['vol_pmt'],      'int8',    os.path.join(mm_path, 'vol_pmt.mmap'))
        vol_led      = create_memmap(f['raw']['vol_led'],      'int8',    os.path.join(mm_path, 'vol_led.mmap'))
    return [vol_time, vol_start, vol_stim_vis, vol_img,
            vol_hifi, vol_stim_aud, vol_flir,
            vol_pmt, vol_led]

In [7]:
import scipy.io as sio
import pandas as pd
import numpy as np
import os

def read_bpod_mat_data(ops, session_start_time, vol_time=None, vol_start=None, vol_stim_vis=None):
    """
    Read bpod session data with optional volume/imaging synchronization.
    
    Parameters:
    - ops: options dict with 'save_path0'
    - session_start_time: reference time for alignment
    - vol_time: volume timestamps (optional, for imaging sync)
    - vol_start: volume start signal (optional, for imaging sync)
    - vol_stim_vis: stimulus visibility signal (optional, for imaging sync)
    """
    
    def _check_keys(d):
        for key in d:
            if isinstance(d[key], sio.matlab.mat_struct):
                d[key] = _todict(d[key])
        return d

    def _todict(matobj):
        d = {}
        for strg in matobj._fieldnames:
            elem = matobj.__dict__[strg]
            if isinstance(elem, sio.matlab.mat_struct):
                d[strg] = _todict(elem)
            elif isinstance(elem, np.ndarray):
                d[strg] = _tolist(elem)
            else:
                d[strg] = elem
        return d

    def _tolist(ndarray):
        elem_list = []
        for sub_elem in ndarray:
            if isinstance(sub_elem, sio.matlab.mat_struct):
                elem_list.append(_todict(sub_elem))
            elif isinstance(sub_elem, np.ndarray):
                elem_list.append(_tolist(sub_elem))
            else:
                elem_list.append(sub_elem)
        return elem_list

    def states_labeling(trial_states):
        if 'Punish' in trial_states.keys() and not np.isnan(trial_states['Punish'][0]):
            outcome = 'punish'
        elif 'Reward' in trial_states.keys() and not np.isnan(trial_states['Reward'][0]):
            outcome = 'reward'
        elif 'PunishNaive' in trial_states.keys() and not np.isnan(trial_states['PunishNaive'][0]):
            outcome = 'naive_punish'
        elif 'RewardNaive' in trial_states.keys() and not np.isnan(trial_states['RewardNaive'][0]):
            outcome = 'naive_reward'
        elif 'DidNotChoose' in trial_states.keys() and not np.isnan(trial_states['DidNotChoose'][0]):
            outcome = 'no_choose'
        else:
            outcome = 'other'
        return outcome

    def get_state(trial_state_dict, target_state, trial_start):
        if target_state in trial_state_dict:
            time_state = 1000 * np.array(trial_state_dict[target_state]) + trial_start
        else:
            time_state = np.array([np.nan, np.nan])
        return time_state

    # Read raw data
    raw = sio.loadmat(
        os.path.join(ops['save_path0'], 'bpod_session_data.mat'),
        struct_as_record=False, squeeze_me=True)
    raw = _check_keys(raw)['SessionData']
    trial_labels = dict()
    n_trials = raw['nTrials']
    trial_states = [raw['RawEvents']['Trial'][ti]['States'] for ti in range(n_trials)]
    trial_events = [raw['RawEvents']['Trial'][ti]['Events'] for ti in range(n_trials)]

    # Trial start and end timestamps
    if vol_start is not None and vol_time is not None:
        # Use volume timing for trial start detection
        trials_start = np.where(np.diff(vol_start == 1))[0]
        trials_start = trials_start[::2]
        trial_labels['time_trial_start'] = vol_time[trials_start]
    else:
        # Use bpod timestamps directly
        trial_labels['time_trial_start'] = 1000 * np.array(raw['TrialStartTimestamp']).reshape(-1)

    trial_labels['time_trial_end'] = 1000 * np.array(raw['TrialEndTimestamp']).reshape(-1)

    # Correct timestamps starting from session start
    trial_labels['time_trial_end'] = trial_labels['time_trial_end'] - trial_labels['time_trial_start'][0] + session_start_time
    trial_labels['time_trial_start'] = trial_labels['time_trial_start'] - trial_labels['time_trial_start'][0] + session_start_time

    # Trial target
    trial_labels['trial_type'] = np.array(raw['TrialTypes']).reshape(-1) - 1
    trial_labels['block_type'] = np.array(raw['BlockTypes']).reshape(-1)

    # Trial outcomes
    trial_labels['outcome'] = np.array([states_labeling(ts) for ts in trial_states], dtype='object')

    # Average ISI
    try:
        mean_short_isi = np.array(raw['TrialSettings'][0]['GUI']['ISIShortMean_s']).reshape(-1)
        mean_long_isi = np.array(raw['TrialSettings'][0]['GUI']['ISILongMean_s']).reshape(-1)
    except Exception:
        mean_short_isi = np.nan
        mean_long_isi = np.nan

    # Trial state timings
    trial_labels['state_window_choice'] = np.array([
        get_state(trial_states[ti], 'WindowChoice', trial_labels['time_trial_start'][ti])
        for ti in range(n_trials)] + ['yicong_forever'], dtype='object')[:-1]
    trial_labels['state_reward'] = np.array([
        get_state(trial_states[ti], 'Reward', trial_labels['time_trial_start'][ti])
        for ti in range(n_trials)] + ['yicong_forever'], dtype='object')[:-1]
    trial_labels['state_punish'] = np.array([
        get_state(trial_states[ti], 'Punish', trial_labels['time_trial_start'][ti])
        for ti in range(n_trials)] + ['yicong_forever'], dtype='object')[:-1]

    # Stimulus timing
    trial_isi = []
    trial_stim_seq = []
    expected_stim = []

    if vol_stim_vis is not None and vol_time is not None:
        # Use volume-based stimulus timing
        indices = np.where(np.diff(vol_stim_vis == 1))[0]
        stim_times = vol_time[indices]
        
        for ti in range(n_trials):
            trial_indices = indices[ti * 4:(ti + 1) * 4]
            if len(trial_indices) == 4:
                trial_times = stim_times[ti * 4:(ti + 1) * 4]
                stim_seq = np.array([[trial_times[0], trial_times[1]],
                                    [trial_times[2], trial_times[3]]])
                isi = (trial_times[2] - trial_times[1])
                
                if trial_labels['trial_type'][ti] == 0:
                    expected = 1000 * mean_long_isi + trial_times[1] 
                elif trial_labels['trial_type'][ti] == 1:
                    expected = 1000 * mean_short_isi + trial_times[1]
                else:
                    expected = np.nan
            else:
                stim_seq = np.array([[np.nan, np.nan], [np.nan, np.nan]])
                isi = np.nan
                expected = np.nan
            
            trial_stim_seq.append(stim_seq)
            trial_isi.append(isi)
            expected_stim.append(expected)
    else:
        # Use bpod event timing
        for ti in range(n_trials):
            if ('BNC1High' in trial_events[ti].keys() and
                'BNC1Low' in trial_events[ti].keys() and
                len(np.array(trial_events[ti]['BNC1High']).reshape(-1)) == 2 and
                len(np.array(trial_events[ti]['BNC1Low']).reshape(-1)) == 2):
                
                stim_seq = 1000 * np.array([trial_events[ti]['BNC1High'], trial_events[ti]['BNC1Low']]) + trial_labels['time_trial_start'][ti]
                stim_seq = np.transpose(stim_seq, [1, 0])
                isi = 1000 * np.array(trial_events[ti]['BNC1High'][1] - trial_events[ti]['BNC1Low'][0])
            else:
                stim_seq = np.array([[np.nan, np.nan], [np.nan, np.nan]])
                isi = np.nan

            if trial_labels['trial_type'][ti] == 0:
                expected = 1000 * np.array(mean_long_isi + trial_events[ti]['BNC1Low'][0]) + trial_labels['time_trial_start'][ti]
            elif trial_labels['trial_type'][ti] == 1:
                expected = 1000 * np.array(mean_short_isi + trial_events[ti]['BNC1Low'][0]) + trial_labels['time_trial_start'][ti]
            else:
                expected = np.nan

            trial_stim_seq.append(stim_seq)
            trial_isi.append(isi)
            expected_stim.append(expected)

    trial_labels['stim_seq'] = np.array(trial_stim_seq + ['yicong_forever'], dtype='object')[:-1]
    trial_labels['isi'] = np.array(trial_isi + ['yicong_forever'], dtype='object')[:-1]
    trial_labels['expected_stim'] = np.array(expected_stim + ['yicong_forever'], dtype='object')[:-1]

    # Licking
    trial_lick = []
    for ti in range(n_trials):
        licking_events = []
        direction = []
        correctness = []
        
        if 'Port1In' in trial_events[ti].keys():
            lick_left = np.array(trial_events[ti]['Port1In']).reshape(-1)
            licking_events.append(lick_left)
            direction.append(np.zeros_like(lick_left))
            if trial_labels['trial_type'][ti] == 0:
                correctness.append(np.ones_like(lick_left))
            else:
                correctness.append(np.zeros_like(lick_left))
        
        if 'Port3In' in trial_events[ti].keys():
            lick_right = np.array(trial_events[ti]['Port3In']).reshape(-1)
            licking_events.append(lick_right)
            direction.append(np.ones_like(lick_right))
            if trial_labels['trial_type'][ti] == 1:
                correctness.append(np.ones_like(lick_right))
            else:
                correctness.append(np.zeros_like(lick_right))
        
        if len(licking_events) > 0:
            licking_events = 1000 * np.concatenate(licking_events).reshape(1, -1) + trial_labels['time_trial_start'][ti]
            direction = np.concatenate(direction).reshape(1, -1)
            correctness = np.concatenate(correctness).reshape(1, -1)
            lick = np.concatenate([licking_events, direction, correctness], axis=0)
            lick = lick[:, np.argsort(lick[0, :])]
            lick = lick[:, lick[0, :] >= trial_labels['state_window_choice'][ti][0]]
            
            if np.size(lick) != 0:
                lick_type = np.full(lick.shape[1], np.nan)
                lick_type[0] = 1
                if (not np.isnan(trial_labels['state_reward'][ti][1]) and
                    len(lick_type) > 1):
                    lick_type[1:][lick[0, 1:] > trial_labels['state_reward'][ti][0]] = 0
                lick_type = lick_type.reshape(1, -1)
                lick = np.concatenate([lick, lick_type], axis=0)
            else:
                lick = np.array([[np.nan], [np.nan], [np.nan], [np.nan]])
        else:
            lick = np.array([[np.nan], [np.nan], [np.nan], [np.nan]])
        
        trial_lick.append(lick)

    trial_labels['lick'] = np.array(trial_lick + ['yicong_forever'], dtype='object')[:-1]

    # Convert to DataFrame
    trial_labels = pd.DataFrame(trial_labels)
    return trial_labels

In [8]:
# remove trial start trigger voltage impulse.
def remove_start_impulse(vol_time, vol_stim_vis):
    min_duration = 100
    changes = np.diff(vol_stim_vis.astype(int))
    start_indices = np.where(changes == 1)[0] + 1
    end_indices = np.where(changes == -1)[0] + 1
    if vol_stim_vis[0] == 1:
        start_indices = np.insert(start_indices, 0, 0)
    if vol_stim_vis[-1] == 1:
        end_indices = np.append(end_indices, len(vol_stim_vis))
    for start, end in zip(start_indices, end_indices):
        duration = vol_time[end-1] - vol_time[start]
        if duration < min_duration:
            vol_stim_vis[start:end] = 0
    return vol_stim_vis

# correct beginning vol_stim_vis if not start from 0.
def correct_vol_start(vol_stim_vis):
    if vol_stim_vis[0] == 1:
        vol_stim_vis[:np.where(vol_stim_vis==0)[0][0]] = 0
    return vol_stim_vis

# detect the rising edge and falling edge of binary series.
def get_trigger_time(
        vol_time,
        vol_bin
        ):
    # find the edge with np.diff and correct it by preappend one 0.
    diff_vol = np.diff(vol_bin, prepend=0)
    idx_up = np.where(diff_vol == 1)[0]
    idx_down = np.where(diff_vol == -1)[0]
    # select the indice for risging and falling.
    # give the edges in ms.
    time_up   = vol_time[idx_up]
    time_down = vol_time[idx_down]
    return time_up, time_down

# find when bpod session timer start.
def get_session_start_time(vol_time, vol_start):
    time_up, _ = get_trigger_time(vol_time, vol_start)
    session_start_time = time_up[0]
    return session_start_time

# correct the fluorescence signal timing.
def correct_time_img_center(time_img):
    # find the frame internal.
    diff_time_img = np.diff(time_img, append=0)
    # correct the last element.
    diff_time_img[-1] = np.mean(diff_time_img[:-1])
    # move the image timing to the center of photon integration interval.
    diff_time_img = diff_time_img / 2
    # correct each individual timing.
    time_neuro = time_img + diff_time_img
    return time_neuro

# save trial neural data.
def save_trials(
        ops, time_neuro, dff, trial_labels,
        vol_time, vol_stim_vis,
        vol_stim_aud, vol_flir,
        vol_pmt, vol_led
        ):
    # file structure:
    # ops['save_path0'] / neural_trials.h5
    # ---- time
    # ---- stim
    # ---- dff
    # ---- vol_stim
    # ---- vol_time
    # trial_labels.csv
    h5_path = os.path.join(ops['save_path0'], 'neural_trials.h5')
    if os.path.exists(h5_path):
        os.remove(h5_path)
    f = h5py.File(h5_path, 'w')
    grp = f.create_group('neural_trials')
    grp['time']         = time_neuro
    grp['dff']          = dff
    grp['vol_time']     = vol_time
    grp['vol_stim_vis'] = vol_stim_vis
    grp['vol_stim_aud'] = vol_stim_aud
    grp['vol_flir']     = vol_flir
    grp['vol_pmt']      = vol_pmt
    grp['vol_led']      = vol_led
    f.close()
    trial_labels.to_csv(os.path.join(ops['save_path0'], 'trial_labels.csv'))

In [9]:
for ops in list_ops:

    print('--- Processing session ---')
    print('Processing session data in:')
    print(ops['save_path0'])
    print('-------------------------------')

    print('Reading dff traces and voltage recordings')
    dff = read_dff(ops)
    [vol_time, vol_start, vol_stim_vis, vol_img,
        vol_hifi, vol_stim_aud, vol_flir,
        vol_pmt, vol_led] = read_raw_voltages(ops)
    vol_stim_vis = remove_start_impulse(vol_time, vol_stim_vis)
    vol_stim_vis = correct_vol_start(vol_stim_vis)
    session_start_time = get_session_start_time(vol_time, vol_start)
    trial_labels = read_bpod_mat_data(ops, session_start_time, vol_time, vol_start, vol_stim_vis)
    print('Correcting 2p camera trigger time')
    # signal trigger time stamps.
    time_img, _   = get_trigger_time(vol_time, vol_img)
    # correct imaging timing.
    time_neuro = correct_time_img_center(time_img)
    # save the final data.
    print('Saving trial data')
    save_trials(
        ops, time_neuro, dff, trial_labels,
        vol_time, vol_stim_vis,
        vol_stim_aud, vol_flir,
        vol_pmt, vol_led)


--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250814_2afc-584
-------------------------------
Reading dff traces and voltage recordings
Correcting 2p camera trigger time
Saving trial data
--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250811_2afc-580
-------------------------------
Reading dff traces and voltage recordings
Correcting 2p camera trigger time
Saving trial data
--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250808_2afc-577
-------------------------------
Reading dff traces and voltage recordings
Correcting 2p camera trigger time
Saving trial data
--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250805_2afc-568
-------------------------------
Reading dff traces and voltage recordings
Correcti

In [10]:
from scipy.signal import savgol_filter

# read trial label csv file into dataframe.
def read_trial_label(ops):
    raw_csv = pd.read_csv(os.path.join(ops['save_path0'], 'trial_labels.csv'), index_col=0)
    # recover object numpy array from csv str.
    def object_parse(k, shape):
        arr = np.array(
            [np.fromstring(s.replace('[', '').replace(']', ''), sep=' ').reshape(shape)
             for s in raw_csv[k].to_list()] + ['yicong_forever'],
            dtype='object')[:-1]
        return arr
    # parse all array.
    time_trial_start = raw_csv['time_trial_start'].to_numpy(dtype='float32')
    time_trial_end = raw_csv['time_trial_end'].to_numpy(dtype='float32')
    trial_type = raw_csv['trial_type'].to_numpy(dtype='int8')
    block_type = raw_csv['block_type'].to_numpy(dtype='int8')
    outcome = raw_csv['outcome'].to_numpy(dtype='object')
    state_window_choice = object_parse('state_window_choice', [-1])
    state_reward = object_parse('state_reward', [-1])
    state_punish = object_parse('state_punish', [-1])
    stim_seq = object_parse('stim_seq', [-1,2])
    expected_stim = object_parse('expected_stim', [-1])
    isi = raw_csv['isi'].to_numpy(dtype='float32')
    lick = object_parse('lick', [4,-1])
    # convert to dataframe.
    trial_labels = pd.DataFrame({
        'time_trial_start': time_trial_start,
        'time_trial_end': time_trial_end,
        'trial_type': trial_type,
        'block_type': block_type,
        'outcome': outcome,
        'state_window_choice': state_window_choice,
        'state_reward': state_reward,
        'state_punish': state_punish,
        'stim_seq': stim_seq,
        'isi': isi,
        'expected_stim': expected_stim,
        'lick': lick,
        })
    return trial_labels

def zscore_normalize(data, axis=1, min_std=1e-8):
    """
    Perform Z-score normalization on 2P neural data.
    
    Parameters:
    - data: NumPy array (n_neurons x n_timepoints) or 1D array for single trace
    - axis: Axis along which to compute mean and std (default: 1 for time axis)
    - min_std: Minimum standard deviation to avoid division by zero
    
    Returns:
    - z_data: Z-score normalized data (same shape as input)
    """
    # Ensure input is a NumPy array
    data = np.asarray(data)
    
    # Handle 1D input (single trace) by reshaping
    if data.ndim == 1:
        data = data.reshape(1, -1)
    
    # Compute mean and standard deviation along specified axis
    mean = np.mean(data, axis=axis, keepdims=True)
    std = np.std(data, axis=axis, keepdims=True)
    
    # Prevent division by zero by setting a minimum std
    std = np.maximum(std, min_std)
    
    # Z-score normalization: (data - mean) / std
    z_data = (data - mean) / std
    
    return z_data

# read trailized neural traces with stimulus alignment.
def read_neural_trials(ops, smooth):
    mm_path, file_path = get_memmap_path(ops, 'neural_trials.h5')
    trial_labels = read_trial_label(ops)
    with h5py.File(file_path, 'r') as f:
        neural_trials = dict()
        dff = np.array(f['neural_trials']['dff'])
        dff = zscore_normalize(dff)
        if smooth:
            window_length=9
            polyorder=3
            dff = np.apply_along_axis(
                savgol_filter, 1, dff,
                window_length=window_length,
                polyorder=polyorder)
        else: pass
        neural_trials['dff']          = create_memmap(dff,                                'float32', os.path.join(mm_path, 'dff.mmap'))
        neural_trials['time']         = create_memmap(f['neural_trials']['time'],         'float32', os.path.join(mm_path, 'time.mmap'))
        neural_trials['trial_labels'] = trial_labels
        neural_trials['vol_time']     = create_memmap(f['neural_trials']['vol_time'],     'float32', os.path.join(mm_path, 'vol_time.mmap'))
        neural_trials['vol_stim_vis'] = create_memmap(f['neural_trials']['vol_stim_vis'], 'int8',    os.path.join(mm_path, 'vol_stim_vis.mmap'))
        neural_trials['vol_stim_aud'] = create_memmap(f['neural_trials']['vol_stim_aud'], 'float32', os.path.join(mm_path, 'vol_stim_aud.mmap'))
        neural_trials['vol_flir']     = create_memmap(f['neural_trials']['vol_flir'],     'int8',    os.path.join(mm_path, 'vol_flir.mmap'))
        neural_trials['vol_pmt']      = create_memmap(f['neural_trials']['vol_pmt'],      'int8',    os.path.join(mm_path, 'vol_pmt.mmap'))
        neural_trials['vol_led']      = create_memmap(f['neural_trials']['vol_led'],      'int8',    os.path.join(mm_path, 'vol_led.mmap'))
    return neural_trials

In [11]:
list_neural_trials = []
for ops in list_ops:
    print('--- Processing session ---')
    print('Processing session data in:')
    print(ops['save_path0'])
    print('-------------------------------')
    print('Reading trailized neural traces with stimulus alignment')
    neural_trials = read_neural_trials(ops, 0)
    list_neural_trials.append(neural_trials)


--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250814_2afc-584
-------------------------------
Reading trailized neural traces with stimulus alignment
--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250811_2afc-580
-------------------------------
Reading trailized neural traces with stimulus alignment
--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250808_2afc-577
-------------------------------
Reading trailized neural traces with stimulus alignment
--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250805_2afc-568
-------------------------------
Reading trailized neural traces with stimulus alignment


# Alignments 


In [12]:
from tqdm import tqdm

# cut sequence into the same length as the shortest one given pivots.
def trim_seq(
        data,
        pivots,
        ):
    if len(data[0].shape) == 1:
        len_l_min = np.min(pivots)
        len_r_min = np.min([len(data[i])-pivots[i] for i in range(len(data))])
        data = [data[i][pivots[i]-len_l_min:pivots[i]+len_r_min]
                for i in range(len(data))]
    if len(data[0].shape) == 3:
        len_l_min = np.min(pivots)
        len_r_min = np.min([len(data[i][0,0,:])-pivots[i] for i in range(len(data))])
        data = [data[i][:, :, pivots[i]-len_l_min:pivots[i]+len_r_min]
                for i in range(len(data))]
    return data


def get_lick_response(
    neural_trials,
    l_frames, r_frames
    ):
    # initialization.
    time = neural_trials['time']
    neu_seq    = []
    neu_time   = []
    direction  = []
    correction = []
    lick_type  = []
    # get all licking events.
    lick = np.concatenate(neural_trials['trial_labels']['lick'].to_numpy(), axis=1)
    # loop over licks.
    for li in tqdm(range(lick.shape[1])):
        t = lick[0,li]
        if not np.isnan(t):
            # get state start timing.
            idx = np.searchsorted(neural_trials['time'], t)
            if idx > l_frames and idx < len(neural_trials['time'])-r_frames:
                # signal response.
                f = neural_trials['dff'][:, idx-l_frames : idx+r_frames]
                f = np.expand_dims(f, axis=0)
                neu_seq.append(f)
                # signal time stamps.
                neu_time.append(neural_trials['time'][idx-l_frames : idx+r_frames] - time[idx])
                # licking properties.
                direction.append(lick[1,li])
                correction.append(lick[2,li])
                lick_type.append(lick[3,li])
    # correct neural data centering at zero.
    neu_time_zero = [np.argmin(np.abs(nt)) for nt in neu_time]
    neu_time = trim_seq(neu_time, neu_time_zero)
    neu_seq = trim_seq(neu_seq, neu_time_zero)
    # concatenate results.
    neu_seq    = np.concatenate(neu_seq, axis=0)
    neu_time   = [nt.reshape(1,-1) for nt in neu_time]
    neu_time   = np.concatenate(neu_time, axis=0)
    direction  = np.array(direction)
    correction = np.array(correction)
    lick_type  = np.array(lick_type)
    # get mean time stamps.
    neu_time = np.mean(neu_time, axis=0)
    # combine results.
    return [neu_seq, neu_time, direction, correction, lick_type]

# extract response around stimulus.
def get_perception_response(
        neural_trials, target_state,
        l_frames, r_frames,
        indices = 0
        ):
    exclude_start_trials = 2
    exclude_end_trials = 2
    # initialization.
    time = neural_trials['time']
    neu_seq    = []
    neu_time   = []
    stim_seq   = []
    stim_value = []
    stim_time  = []
    led_value  = []
    trial_type = []
    block_type = []
    isi        = []
    decision   = []
    outcome    = []
    # loop over trials.
    for ti in tqdm(range(len(neural_trials['trial_labels']))):
        t = neural_trials['trial_labels'][target_state][ti].flatten()[indices]
        if (not np.isnan(t) and
            ti >= exclude_start_trials and
            ti < len(neural_trials['trial_labels'])-exclude_end_trials
            ):
            # get state start timing.
            idx = np.searchsorted(neural_trials['time'], t)
            if idx > l_frames and idx < len(neural_trials['time'])-r_frames:
                # signal response.
                f = neural_trials['dff'][:, idx-l_frames : idx+r_frames]
                f = np.expand_dims(f, axis=0)
                neu_seq.append(f)
                # signal time stamps.
                neu_time.append(neural_trials['time'][idx-l_frames : idx+r_frames] - time[idx])
                # voltage.
                vol_t_c = np.searchsorted(neural_trials['vol_time'], neural_trials['time'][idx])
                vol_t_l = np.searchsorted(neural_trials['vol_time'], neural_trials['time'][idx-l_frames])
                vol_t_r = np.searchsorted(neural_trials['vol_time'], neural_trials['time'][idx+r_frames])
                stim_time.append(neural_trials['vol_time'][vol_t_l:vol_t_r] - neural_trials['vol_time'][vol_t_c])
                stim_value.append(neural_trials['vol_stim_vis'][vol_t_l:vol_t_r])
                led_value.append(neural_trials['vol_led'][vol_t_l:vol_t_r])
                # task variables.
                stim_seq.append(neural_trials['trial_labels']['stim_seq'][ti].reshape(1,2,2) - t)
                trial_type.append(neural_trials['trial_labels']['trial_type'][ti])
                block_type.append(neural_trials['trial_labels']['block_type'][ti])
                # isi.
                isi.append(neural_trials['trial_labels']['isi'][ti])
                decision.append(neural_trials['trial_labels']['lick'][ti][1,0])
                outcome.append(neural_trials['trial_labels']['outcome'][ti])
        else: pass
    # correct neural data centering at zero.
    neu_time_zero = [np.argmin(np.abs(nt)) for nt in neu_time]
    neu_time = trim_seq(neu_time, neu_time_zero)
    neu_seq = trim_seq(neu_seq, neu_time_zero)
    # correct voltage data centering at zero.
    stim_time_zero = [np.argmin(np.abs(sv)) for sv in stim_value]
    stim_time = trim_seq(stim_time, stim_time_zero)
    stim_value = trim_seq(stim_value, stim_time_zero)
    led_value = trim_seq(led_value, stim_time_zero)
    # concatenate results.
    neu_seq    = np.concatenate(neu_seq, axis=0)
    neu_time   = [nt.reshape(1,-1) for nt in neu_time]
    neu_time   = np.concatenate(neu_time, axis=0)
    stim_seq   = np.concatenate(stim_seq, axis=0)
    stim_value = [sv.reshape(1,-1) for sv in stim_value]
    stim_value = np.concatenate(stim_value, axis=0)
    stim_time  = [st.reshape(1,-1) for st in stim_time]
    stim_time  = np.concatenate(stim_time, axis=0)
    led_value  = [lv.reshape(1,-1) for lv in led_value]
    led_value  = np.concatenate(led_value, axis=0)
    trial_type = np.array(trial_type)
    block_type = np.array(block_type)
    isi        = np.array(isi)
    decision   = np.array(decision)
    outcome    = np.array(outcome)
    # get mean time stamps.aa
    neu_time  = np.mean(neu_time, axis=0)
    stim_time = np.mean(stim_time, axis=0)
    # combine results.
    return [neu_seq, neu_time, stim_seq, stim_value, stim_time, led_value, trial_type, block_type, isi, decision, outcome]


## Alignemnt Figures

## Pooling sessions

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import sem
import pandas as pd
from dash import Dash, dcc, html, callback, Input, Output, State
import dash_bootstrap_components as dbc

# ============================================
# HELPER FUNCTIONS AND DATASTORAGE CLASS
# ============================================

def get_early_late_epochs(block_type, early_n=5, late_n=5):
    """Label early and late trials in blocks of type 1 and 2."""
    block_type = np.array(block_type)
    labels = np.zeros_like(block_type, dtype=int)
    
    diff = np.diff(block_type, prepend=block_type[0], append=block_type[-1])
    block_starts = np.where(diff != 0)[0]
    block_ends = np.where(diff != 0)[0][1:]
    block_ends = np.append(block_ends, len(block_type))
    
    for start, end in zip(block_starts, block_ends):
        block_id = block_type[start]
        if block_id not in [1, 2]:
            continue
        block_length = end - start
        if block_length < early_n:
            labels[start:end] = 1
        else:
            labels[start:start + early_n] = 1
            if block_length >= early_n + late_n:
                labels[end - late_n:end] = 2
            elif block_length > early_n:
                labels[end - (block_length - early_n):end] = 2
    return labels

def compute_avg_and_sem(data, mask, axis=(0, 1)):
    """Compute mean, SEM, and trial × neuron count for masked data."""
    masked_data = data[mask, :, :]
    n_trials, n_neurons, _ = masked_data.shape
    avg = np.nanmean(masked_data, axis=axis)
    sem_data = sem(masked_data, axis=axis, nan_policy='omit')
    return avg, sem_data, n_trials * n_neurons

def get_y_limits(avg_data1, sem_data1, avg_data2, sem_data2):
    """Compute y-axis limits from average and SEM data."""
    if np.all(np.isnan(avg_data1)) and np.all(np.isnan(avg_data2)):
        return np.nan, np.nan
    y_min1 = np.nanmin(avg_data1 - sem_data1) if not np.all(np.isnan(avg_data1)) else np.nan
    y_max1 = np.nanmax(avg_data1 + sem_data1) if not np.all(np.isnan(avg_data1)) else np.nan
    y_min2 = np.nanmin(avg_data2 - sem_data2) if not np.all(np.isnan(avg_data2)) else np.nan
    y_max2 = np.nanmax(avg_data2 + sem_data2) if not np.all(np.isnan(avg_data2)) else np.nan
    y_min = np.nanmin([y_min1, y_min2])
    y_max = np.nanmax([y_max1, y_max2])
    return y_min, y_max

def process_trial_data(neu_seq, neu_time, stim_seq, trial_type, block_type, outcome, 
                      trial_mask1, trial_mask2, label1, label2, stim_mask, target_state, indices):
    """Process neural data and stimulus timings for a given alignment and trial conditions."""
    avg_data1, sem_data1, n_trials_neurons1 = compute_avg_and_sem(neu_seq, trial_mask1)
    avg_data2, sem_data2, n_trials_neurons2 = compute_avg_and_sem(neu_seq, trial_mask2)
    
    relevant_trials = trial_mask1 | trial_mask2
    if target_state == 'stim_seq':
        stim_idx = indices // 2
        alignment_times = stim_seq[relevant_trials, stim_idx, 0]
        adjusted_stim_seq = stim_seq[relevant_trials, :, :] - alignment_times[:, None, None]
        mean_stim_times = np.nanmean(adjusted_stim_seq, axis=0)
    else:
        mean_stim_times = np.nanmean(stim_seq[relevant_trials & stim_mask, :, :], axis=0)
    
    return neu_time, avg_data1, sem_data1, n_trials_neurons1, avg_data2, sem_data2, n_trials_neurons2, mean_stim_times

def process_licking_data(licks_per_trial, correctness_per_trial, trial_mask, l_frames, r_frames, frame_rate=30):
    """Process licking data for a given condition."""
    time_window_ms = (l_frames + r_frames) / frame_rate * 1000
    dt_ms = 1000 / frame_rate
    time_bins = np.arange(-l_frames, r_frames) * dt_ms
    
    correct_lick_counts = np.zeros((np.sum(trial_mask), len(time_bins) - 1))
    incorrect_lick_counts = np.zeros((np.sum(trial_mask), len(time_bins) - 1))
    
    trial_idx = 0
    for trial in range(len(trial_mask)):
        if not trial_mask[trial]:
            continue
        
        lick_times = licks_per_trial[trial]
        correctness = correctness_per_trial[trial]
        
        if len(lick_times) == 0:
            trial_idx += 1
            continue
        
        correct_licks = lick_times[correctness == 1]
        incorrect_licks = lick_times[correctness == 0]
        
        if len(correct_licks) > 0:
            correct_lick_counts[trial_idx, :], _ = np.histogram(correct_licks, bins=time_bins)
        
        if len(incorrect_licks) > 0:
            incorrect_lick_counts[trial_idx, :], _ = np.histogram(incorrect_licks, bins=time_bins)
        
        trial_idx += 1
    
    dt_sec = dt_ms / 1000
    correct_lick_rates = correct_lick_counts / dt_sec
    incorrect_lick_rates = incorrect_lick_counts / dt_sec
    
    avg_correct = np.nanmean(correct_lick_rates, axis=0)
    sem_correct = sem(correct_lick_rates, axis=0, nan_policy='omit')
    n_correct = np.sum(trial_mask)
    
    avg_incorrect = np.nanmean(incorrect_lick_rates, axis=0)
    sem_incorrect = sem(incorrect_lick_rates, axis=0, nan_policy='omit')
    n_incorrect = np.sum(trial_mask)
    
    lick_time = time_bins[:-1]
    
    return lick_time, avg_correct, sem_correct, n_correct, avg_incorrect, sem_incorrect, n_incorrect

def pool_session_data(neural_trials_list, labels_list, state, l_frames, r_frames, indices, epoch_list=None):
    """
    Pool data from multiple sessions, including neural and licking data.
    
    Args:
        neural_trials_list: List of session data (neural_trials).
        labels_list: List of neuron labels per session (e.g., neuron IDs or indices).
        state: Alignment state (e.g., 'time_trial_start', 'state_reward').
        l_frames, r_frames: Frames before/after alignment point.
        indices: Index for alignment (e.g., 0 for Stim 1, 2 for Stim 2).
        epoch_list: List of precomputed epoch arrays per session (optional).
    
    Returns:
        tuple: Pooled neu_seq, neu_time, trial_type, block_type, isi, decision, labels, outcomes, stim_seq, epoch (if provided), licks_per_trial, correctness_per_trial
    """
    neu_seqs = []
    stim_seqs = []
    trial_types = []
    block_types = []
    isis = []
    decisions = []
    all_labels = []
    all_outcomes = []
    all_epochs = [] if epoch_list is not None else None
    licks_per_trials = []
    correctness_per_trials = []

    neuron_counts = [labels.shape[0] for labels in labels_list]
    total_neurons = sum(neuron_counts)
    neuron_offset = 0

    for i, (neural_trials, session_labels) in enumerate(zip(neural_trials_list, labels_list)):
        neu_seq, neu_time, stim_seq, stim_value, stim_time, led_value, trial_type, block_type, isi, decision, outcome = get_perception_response(
            neural_trials, state, l_frames, r_frames, indices=indices)

        n_trials, n_neurons, n_time = neu_seq.shape
        padded_neu_seq = np.zeros((n_trials, total_neurons, n_time))
        padded_neu_seq[:, neuron_offset:neuron_offset + n_neurons, :] = neu_seq

        # Get licking data, accounting for intrinsic exclusion of 2 trials from start/end
        trial_labels = neural_trials['trial_labels']
        outcome_all = trial_labels['outcome']  # 'reward' or 'punish'
        n_trials_total = len(trial_labels['lick'])
        trial_indices_to_use = np.arange(2, n_trials_total - 2) if n_trials_total >= 4 else np.array([])
        licks_per_trial_s = []
        correctness_per_trial_s = []

        for trial in range(n_trials_total):
            if state == 'state_outcome':
                start_time_alignment_reward = neural_trials['trial_labels'][state][trial].flatten()[indices]
                start_time_alignment_punish = neural_trials['trial_labels'][state][trial].flatten()[indices]
                start_time_alignment = start_time_alignment_reward if outcome_all[trial] == 'reward' else start_time_alignment_punish
            else:
                start_time_alignment = neural_trials['trial_labels'][state][trial].flatten()[indices]
            lick_times = trial_labels['lick'][trial][0] - start_time_alignment
            correctness = trial_labels['lick'][trial][2]
            if trial in trial_indices_to_use:
                licks_per_trial_s.append(lick_times)
                correctness_per_trial_s.append(correctness)

        neu_seqs.append(padded_neu_seq)
        stim_seqs.append(stim_seq)
        trial_types.append(trial_type)
        block_types.append(block_type)
        isis.append(isi)
        decisions.append(decision)
        all_labels.append(session_labels)
        all_outcomes.append(outcome)
        licks_per_trials.append(licks_per_trial_s)
        correctness_per_trials.append(correctness_per_trial_s)
        if epoch_list is not None:
            all_epochs.append(epoch_list[i])

        neuron_offset += n_neurons

    pooled_neu_seq = np.concatenate(neu_seqs, axis=0)
    pooled_stim_seq = np.concatenate(stim_seqs, axis=0)
    pooled_trial_type = np.concatenate(trial_types, axis=0)
    pooled_block_type = np.concatenate(block_types, axis=0)
    pooled_isi = np.concatenate(isis, axis=0)
    pooled_decision = np.concatenate(decisions, axis=0)
    pooled_labels = np.concatenate(all_labels, axis=0)
    pooled_outcomes = np.concatenate(all_outcomes, axis=0)
    pooled_licks_per_trial = [lick for session_licks in licks_per_trials for lick in session_licks]
    pooled_correctness_per_trial = [corr for session_corr in correctness_per_trials for corr in session_corr]

    if epoch_list is not None:
        pooled_epoch = np.concatenate(all_epochs, axis=0)
        return (pooled_neu_seq, neu_time, pooled_trial_type, pooled_block_type, pooled_isi, pooled_decision,
                pooled_labels, pooled_outcomes, pooled_stim_seq, pooled_epoch, pooled_licks_per_trial, pooled_correctness_per_trial)
    return (pooled_neu_seq, neu_time, pooled_trial_type, pooled_block_type, pooled_isi, pooled_decision,
            pooled_labels, pooled_outcomes, pooled_stim_seq, pooled_licks_per_trial, pooled_correctness_per_trial)

class DataStorage:
    """Store precomputed data for each alignment to avoid recalculation."""
    def __init__(self):
        self.data_cache = {}
    
    def compute_data(self, neural_trials_list, l_frames=60, r_frames=120):
        """Precompute all alignment data for both neural and licking across multiple sessions."""
        alignments = [
            ('time_trial_start', 0, 'Trial Start'),
            ('stim_seq', 0, 'Stim 1 Onset'),
            ('stim_seq', 2, 'Stim 2 Onset'),
            ('expected_stim', 0, 'Expected Stimulus'),
            ('state_window_choice', 0, 'Servo In'),
            ('state_window_choice', 1, 'Choice'),
            ('state_outcome', 0, 'Outcome'),
            ('time_trial_end', 0, 'Trial End'),
        ]
        
        row_conditions = [
            ('Short trials', lambda st, rw, pu, sb, lb, ee, le: (st & rw, st & pu, 'Rewarded', 'Punished', st)),
            ('Long trials', lambda st, rw, pu, sb, lb, ee, le: ((~st) & rw, (~st) & pu, 'Rewarded', 'Punished', ~st)),
            ('Short standard trials', lambda st, rw, pu, sb, lb, ee, le: (sb & st & rw, sb & st & pu, 'Rewarded', 'Punished', st)),
            ('Short rare trials', lambda st, rw, pu, sb, lb, ee, le: (lb & st & rw, lb & st & pu, 'Rewarded', 'Punished', st)),
            ('Long standard trials', lambda st, rw, pu, sb, lb, ee, le: (lb & (~st) & rw, lb & (~st) & pu, 'Rewarded', 'Punished', ~st)),
            ('Long rare trials', lambda st, rw, pu, sb, lb, ee, le: (sb & (~st) & rw, sb & (~st) & pu, 'Rewarded', 'Punished', ~st)),
            ('Early standard short trials', lambda st, rw, pu, sb, lb, ee, le: (ee & sb & st & rw, ee & sb & st & pu, 'Rewarded', 'Punished', st)),
            ('Late standard short trials', lambda st, rw, pu, sb, lb, ee, le: (le & sb & st & rw, le & sb & st & pu, 'Rewarded', 'Punished', st)),
            ('Early rare short trials', lambda st, rw, pu, sb, lb, ee, le: (ee & lb & st & rw, ee & lb & st & pu, 'Rewarded', 'Punished', st)),
            ('Late rare short trials', lambda st, rw, pu, sb, lb, ee, le: (le & lb & st & rw, le & lb & st & pu, 'Rewarded', 'Punished', st)),
            ('Early standard long trials', lambda st, rw, pu, sb, lb, ee, le: (ee & lb & (~st) & rw, ee & lb & (~st) & pu, 'Rewarded', 'Punished', ~st)),
            ('Late standard long trials', lambda st, rw, pu, sb, lb, ee, le: (le & lb & (~st) & rw, le & lb & (~st) & pu, 'Rewarded', 'Punished', ~st)),
            ('Early rare long trials', lambda st, rw, pu, sb, lb, ee, le: (ee & sb & (~st) & rw, ee & sb & (~st) & pu, 'Rewarded', 'Punished', ~st)),
            ('Late rare long trials', lambda st, rw, pu, sb, lb, ee, le: (le & sb & (~st) & rw, le & sb & (~st) & pu, 'Rewarded', 'Punished', ~st)),
        ]

        # Handle single session compatibility
        list_neural_trials = neural_trials_list if isinstance(neural_trials_list, list) else [neural_trials_list]

        # Precompute epochs and metadata per session
        precomputed = []
        labels_list = []
        for session in list_neural_trials:
            _, _, _, _, _, _, trial_type_s, block_type_s, _, _, outcome_s = get_perception_response(
                session, 'time_trial_start', l_frames, r_frames, indices=0)
            epoch_s = get_early_late_epochs(block_type_s, early_n=5, late_n=5)
            # Placeholder labels (adjust if actual neuron labels available)
            labels_s = np.arange(trial_type_s.shape[0])
            precomputed.append({
                'epoch': epoch_s,
                'outcome': outcome_s,
                'trial_type': trial_type_s,
                'block_type': block_type_s
            })
            labels_list.append(labels_s)

        for target_state, indices, align_label in alignments:
            if target_state == 'state_outcome':
                # Pool reward trials
                neu_seq_r, neu_time, trial_type_r, block_type_r, isi_r, decision_r, labels_r, outcome_r, stim_seq_r, epoch_r, licks_r, corr_r = pool_session_data(
                    list_neural_trials, labels_list, 'state_reward', l_frames, r_frames, indices=0, epoch_list=[p['epoch'] for p in precomputed])
                reward_indices = [np.where(p['outcome'] == 'reward')[0] for p in precomputed]
                epoch_r = np.concatenate([p['epoch'][idx] for p, idx in zip(precomputed, reward_indices)], axis=0)
                
                # Pool punish trials
                neu_seq_p, _, trial_type_p, block_type_p, isi_p, decision_p, labels_p, outcome_p, stim_seq_p, epoch_p, licks_p, corr_p = pool_session_data(
                    list_neural_trials, labels_list, 'state_punish', l_frames, r_frames, indices=0, epoch_list=[p['epoch'] for p in precomputed])
                punish_indices = [np.where(p['outcome'] == 'punish')[0] for p in precomputed]
                epoch_p = np.concatenate([p['epoch'][idx] for p, idx in zip(precomputed, punish_indices)], axis=0)
                
                # Concatenate reward and punish
                neu_seq = np.concatenate((neu_seq_r, neu_seq_p), axis=0)
                stim_seq = np.concatenate((stim_seq_r, stim_seq_p), axis=0)
                trial_type = np.concatenate((trial_type_r, trial_type_p), axis=0)
                block_type = np.concatenate((block_type_r, block_type_p), axis=0)
                outcome = np.concatenate((outcome_r, outcome_p), axis=0)
                epoch = np.concatenate((epoch_r, epoch_p), axis=0)
                licks_per_trial = licks_r + licks_p
                correctness_per_trial = corr_r + corr_p
            else:
                # Standard alignment
                neu_seq, neu_time, trial_type, block_type, isi, decision, labels, outcome, stim_seq, epoch, licks_per_trial, correctness_per_trial = pool_session_data(
                    list_neural_trials, labels_list, target_state, l_frames, r_frames, indices=indices, epoch_list=[p['epoch'] for p in precomputed])

            early_epoch = epoch == 1
            late_epoch = epoch == 2
            short_trials = trial_type == 0
            long_trials = trial_type == 1
            rewarded_trials = outcome == 'reward'
            punished_trials = outcome == 'punish'
            short_block = block_type == 1
            long_block = block_type == 2

            self.data_cache[align_label] = {
                'neu_seq': neu_seq,
                'neu_time': neu_time,
                'stim_seq': stim_seq,
                'trial_type': trial_type,
                'block_type': block_type,
                'outcome': outcome,
                'short_trials': short_trials,
                'long_trials': long_trials,
                'rewarded_trials': rewarded_trials,
                'punished_trials': punished_trials,
                'short_block': short_block,
                'long_block': long_block,
                'early_epoch': early_epoch,
                'late_epoch': late_epoch,
                'licks_per_trial': licks_per_trial,
                'correctness_per_trial': correctness_per_trial,
                'row_conditions': row_conditions,
                'target_state': target_state,
                'indices': indices,
                'align_label': align_label,
            }
    def get_figure(self, alignment, selected_conditions, outcomes, mode='separate'):
        """Generate figure for given alignment and conditions with neural + licking data."""
        if alignment not in self.data_cache:
            return go.Figure().add_annotation(text='No data available')
        
        data = self.data_cache[alignment]
        
        y_mins_neural = []
        y_maxs_neural = []
        condition_data = {}
        overall_relevant = np.zeros_like(data['short_trials'], dtype=bool)
        
        for condition in selected_conditions:
            mask_func = next(func for title, func in data['row_conditions'] if title == condition)
            mask1, mask2, label1, label2, stim_mask = mask_func(
                data['short_trials'], data['rewarded_trials'], data['punished_trials'],
                data['short_block'], data['long_block'], data['early_epoch'], data['late_epoch'])
            
            if 'rewarded' not in outcomes:
                mask1 = np.zeros_like(mask1, dtype=bool)
            if 'punished' not in outcomes:
                mask2 = np.zeros_like(mask2, dtype=bool)
            
            neu_time, avg_data1, sem_data1, n_trials_neurons1, avg_data2, sem_data2, n_trials_neurons2, mean_stim_times = process_trial_data(
                data['neu_seq'], data['neu_time'], data['stim_seq'], data['trial_type'], data['block_type'], data['outcome'],
                mask1, mask2, label1, label2, stim_mask, data['target_state'], data['indices']
            )
            
            lick_time, avg_correct, sem_correct, n_correct, avg_incorrect, sem_incorrect, n_incorrect = process_licking_data(
                data['licks_per_trial'], data['correctness_per_trial'], mask1, l_frames=60, r_frames=120)
            
            condition_data[condition] = {
                'neu_time': neu_time,
                'avg_data1': avg_data1, 'sem_data1': sem_data1, 'n1': n_trials_neurons1, 'label1': label1,
                'avg_data2': avg_data2, 'sem_data2': sem_data2, 'n2': n_trials_neurons2, 'label2': label2,
                'mean_stim_times': mean_stim_times,
                'lick_time': lick_time,
                'avg_correct': avg_correct, 'sem_correct': sem_correct, 'n_correct': n_correct,
                'avg_incorrect': avg_incorrect, 'sem_incorrect': sem_incorrect, 'n_incorrect': n_incorrect,
            }
            
            overall_relevant = overall_relevant | (mask1 | mask2)
            
            y_min, y_max = get_y_limits(avg_data1, sem_data1, avg_data2, sem_data2)
            if not np.isnan(y_min):
                y_mins_neural.append(y_min)
            if not np.isnan(y_max):
                y_maxs_neural.append(y_max)
        
        if not y_mins_neural or not y_maxs_neural:
            y_lim_neural = (-0.1, 0.1)
        else:
            y_min = min(y_mins_neural)
            y_max = max(y_maxs_neural)
            y_range = y_max - y_min
            y_lim_neural = (y_min - 0.1 * y_range, y_max + 0.1 * y_range) if y_range > 0 else (-0.1, 0.1)
        
        if np.any(overall_relevant):
            if data['target_state'] == 'stim_seq':
                stim_idx = data['indices'] // 2
                alignment_times = data['stim_seq'][overall_relevant, stim_idx, 0]
                adjusted_stim_seq = data['stim_seq'][overall_relevant, :, :] - alignment_times[:, None, None]
                overall_mean_stim_times = np.nanmean(adjusted_stim_seq, axis=0)
            else:
                overall_stim_mask = np.ones_like(overall_relevant, dtype=bool)
                overall_mean_stim_times = np.nanmean(data['stim_seq'][overall_relevant & overall_stim_mask, :, :], axis=0)
        else:
            overall_mean_stim_times = np.full((2, 2), np.nan)
        
        if mode == 'superimposed':
            fig = make_subplots(
                rows=2, cols=1,
                shared_xaxes=True,
                vertical_spacing=0.15,
                row_heights=[0.5, 0.5],
                subplot_titles=('Neural Activity', 'Licking Rate')
            )
            
            dash_styles = ['solid', 'dot', 'dash', 'longdash', 'dashdot', 'longdashdot']
            
            for i, condition in enumerate(selected_conditions):
                cdata = condition_data[condition]
                dash = dash_styles[i % len(dash_styles)]
                
                if 'rewarded' in outcomes and not np.all(np.isnan(cdata['avg_data1'])):
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data1'] + cdata['sem_data1'],
                            mode='lines',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=1, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data1'] - cdata['sem_data1'],
                            mode='lines',
                            fill='tonexty',
                            fillcolor='rgba(0, 255, 0, 0.3)',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=1, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data1'],
                            mode='lines',
                            name=f"{condition} - Neural {cdata['label1']} (n={cdata['n1']})",
                            line=dict(color='green', width=2, dash=dash),
                            hovertemplate='<b>%{fullData.name}</b><br>Time: %{x:.1f} ms<br>Mean: %{y:.3f}<extra></extra>',
                            legendgroup=f'neural_{i}'
                        ),
                        row=1, col=1
                    )
                
                if 'punished' in outcomes and not np.all(np.isnan(cdata['avg_data2'])):
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data2'] + cdata['sem_data2'],
                            mode='lines',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=1, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data2'] - cdata['sem_data2'],
                            mode='lines',
                            fill='tonexty',
                            fillcolor='rgba(255, 0, 0, 0.3)',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=1, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data2'],
                            mode='lines',
                            name=f"{condition} - Neural {cdata['label2']} (n={cdata['n2']})",
                            line=dict(color='red', width=2, dash=dash),
                            hovertemplate='<b>%{fullData.name}</b><br>Time: %{x:.1f} ms<br>Mean: %{y:.3f}<extra></extra>',
                            legendgroup=f'neural_{i}'
                        ),
                        row=1, col=1
                    )
                
                if not np.all(np.isnan(cdata['avg_correct'])):
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_correct'] + cdata['sem_correct'],
                            mode='lines',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=2, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_correct'] - cdata['sem_correct'],
                            mode='lines',
                            fill='tonexty',
                            fillcolor='rgba(0, 255, 0, 0.3)',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=2, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_correct'],
                            mode='lines',
                            name=f"{condition} - Correct Licks (n={cdata['n_correct']})",
                            line=dict(color='green', width=2, dash=dash),
                            hovertemplate='<b>%{fullData.name}</b><br>Time: %{x:.1f} ms<br>Rate: %{y:.2f} licks/s<extra></extra>',
                            legendgroup=f'lick_{i}'
                        ),
                        row=2, col=1
                    )
                
                if not np.all(np.isnan(cdata['avg_incorrect'])):
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_incorrect'] + cdata['sem_incorrect'],
                            mode='lines',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=2, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_incorrect'] - cdata['sem_incorrect'],
                            mode='lines',
                            fill='tonexty',
                            fillcolor='rgba(255, 0, 0, 0.3)',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=2, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_incorrect'],
                            mode='lines',
                            name=f"{condition} - Incorrect Licks (n={cdata['n_incorrect']})",
                            line=dict(color='red', width=2, dash=dash),
                            hovertemplate='<b>%{fullData.name}</b><br>Time: %{x:.1f} ms<br>Rate: %{y:.2f} licks/s<extra></extra>',
                            legendgroup=f'lick_{i}'
                        ),
                        row=2, col=1
                    )
            
            if not np.all(np.isnan(overall_mean_stim_times)):
                for stim_idx in range(2):
                    start_time = overall_mean_stim_times[stim_idx, 0]
                    end_time = overall_mean_stim_times[stim_idx, 1]
                    if np.isfinite(start_time) and np.isfinite(end_time):
                        for row in [1, 2]:
                            fig.add_shape(
                                type='rect',
                                x0=start_time,
                                x1=end_time,
                                y0=0 if row == 2 else y_lim_neural[0],
                                y1=10 if row == 2 else y_lim_neural[1],
                                fillcolor='blue' if stim_idx == 0 else 'purple',
                                opacity=0.2,
                                layer='below',
                                line_width=0,
                            )
            
            fig.update_layout(
                title=f'Neural and Licking Data - {alignment}',
                height=600,
                showlegend=True,
                template='plotly_white',
            )
            fig.update_xaxes(title_text='Time (ms)', row=2, col=1)
            fig.update_yaxes(title_text='Neural Activity', row=1, col=1, range=y_lim_neural)
            fig.update_yaxes(title_text='Lick Rate (licks/s)', row=2, col=1, range=[0, 10])
        
        else:  # mode == 'separate'
            n_conditions = len(selected_conditions)
            fig = make_subplots(
                rows=n_conditions * 2,
                cols=1,
                shared_xaxes=True,
                vertical_spacing=0.1,
                row_heights=[0.5 / n_conditions] * (n_conditions * 2),
                subplot_titles=[f"{cond} - Neural Activity" if i % 2 == 0 else f"{cond} - Licking Rate"
                                for i, cond in enumerate(selected_conditions * 2)]
            )
            
            for i, condition in enumerate(selected_conditions):
                cdata = condition_data[condition]
                row_neural = i * 2 + 1
                row_lick = i * 2 + 2
                
                if 'rewarded' in outcomes and not np.all(np.isnan(cdata['avg_data1'])):
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data1'] + cdata['sem_data1'],
                            mode='lines',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_neural, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data1'] - cdata['sem_data1'],
                            mode='lines',
                            fill='tonexty',
                            fillcolor='rgba(0, 255, 0, 0.3)',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_neural, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data1'],
                            mode='lines',
                            name=f"Neural {cdata['label1']} (n={cdata['n1']})",
                            line=dict(color='green', width=2),
                            hovertemplate='<b>%{fullData.name}</b><br>Time: %{x:.1f} ms<br>Mean: %{y:.3f}<extra></extra>',
                            legendgroup=f'neural_{i}'
                        ),
                        row=row_neural, col=1
                    )
                
                if 'punished' in outcomes and not np.all(np.isnan(cdata['avg_data2'])):
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data2'] + cdata['sem_data2'],
                            mode='lines',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_neural, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data2'] - cdata['sem_data2'],
                            mode='lines',
                            fill='tonexty',
                            fillcolor='rgba(255, 0, 0, 0.3)',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_neural, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['neu_time'],
                            y=cdata['avg_data2'],
                            mode='lines',
                            name=f"Neural {cdata['label2']} (n={cdata['n2']})",
                            line=dict(color='red', width=2),
                            hovertemplate='<b>%{fullData.name}</b><br>Time: %{x:.1f} ms<br>Mean: %{y:.3f}<extra></extra>',
                            legendgroup=f'neural_{i}'
                        ),
                        row=row_neural, col=1
                    )
                
                if not np.all(np.isnan(cdata['avg_correct'])):
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_correct'] + cdata['sem_correct'],
                            mode='lines',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_lick, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_correct'] - cdata['sem_correct'],
                            mode='lines',
                            fill='tonexty',
                            fillcolor='rgba(0, 255, 0, 0.3)',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_lick, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_correct'],
                            mode='lines',
                            name=f"Correct Licks (n={cdata['n_correct']})",
                            line=dict(color='green', width=2),
                            hovertemplate='<b>%{fullData.name}</b><br>Time: %{x:.1f} ms<br>Rate: %{y:.2f} licks/s<extra></extra>',
                            legendgroup=f'lick_{i}'
                        ),
                        row=row_lick, col=1
                    )
                
                if not np.all(np.isnan(cdata['avg_incorrect'])):
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_incorrect'] + cdata['sem_incorrect'],
                            mode='lines',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_lick, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_incorrect'] - cdata['sem_incorrect'],
                            mode='lines',
                            fill='tonexty',
                            fillcolor='rgba(255, 0, 0, 0.3)',
                            line=dict(color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_lick, col=1
                    )
                    fig.add_trace(
                        go.Scatter(
                            x=cdata['lick_time'],
                            y=cdata['avg_incorrect'],
                            mode='lines',
                            name=f"Incorrect Licks (n={cdata['n_incorrect']})",
                            line=dict(color='red', width=2),
                            hovertemplate='<b>%{fullData.name}</b><br>Time: %{x:.1f} ms<br>Rate: %{y:.2f} licks/s<extra></extra>',
                            legendgroup=f'lick_{i}'
                        ),
                        row=row_lick, col=1
                    )
                
                if not np.all(np.isnan(cdata['mean_stim_times'])):
                    for stim_idx in range(2):
                        start_time = cdata['mean_stim_times'][stim_idx, 0]
                        end_time = cdata['mean_stim_times'][stim_idx, 1]
                        if np.isfinite(start_time) and np.isfinite(end_time):
                            fig.add_shape(
                                type='rect',
                                x0=start_time,
                                x1=end_time,
                                y0=y_lim_neural[0],
                                y1=y_lim_neural[1],
                                fillcolor='blue' if stim_idx == 0 else 'purple',
                                opacity=0.2,
                                layer='below',
                                line_width=0,
                                row=row_neural,
                                col=1
                            )
                            fig.add_shape(
                                type='rect',
                                x0=start_time,
                                x1=end_time,
                                y0=0,
                                y1=10,
                                fillcolor='blue' if stim_idx == 0 else 'purple',
                                opacity=0.2,
                                layer='below',
                                line_width=0,
                                row=row_lick,
                                col=1
                            )
            
            fig.update_layout(
                title=f'Neural and Licking Data - {alignment}',
                height=300 * n_conditions,
                showlegend=True,
                template='plotly_white',
            )
            for i in range(n_conditions):
                fig.update_yaxes(title_text='Neural Activity', row=i * 2 + 1, col=1, range=y_lim_neural)
                fig.update_yaxes(title_text='Lick Rate (licks/s)', row=i * 2 + 2, col=1, range=[0, 10])
            fig.update_xaxes(title_text='Time (ms)', row=n_conditions * 2, col=1)
        
        return fig

# Initialize data storage
data_storage = DataStorage()

# Verify get_figure exists
print("Checking DataStorage methods:", [method for method in dir(data_storage) if not method.startswith('__')])

# Initialize Dash app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Define app layout
app.layout = dbc.Container([
    dbc.Row([
        dbc.Col([
            html.H1('Interactive Neural Traces Viewer', className='text-center my-4 text-primary')
        ], width=12)
    ]),
    
    dbc.Row([
        dbc.Col([
            html.Label('Select Alignment:', className='fw-bold mb-2'),
            dcc.Dropdown(
                id='alignment-selector',
                options=[
                    {'label': 'Trial Start', 'value': 'Trial Start'},
                    {'label': 'Stim 1 Onset', 'value': 'Stim 1 Onset'},
                    {'label': 'Stim 2 Onset', 'value': 'Stim 2 Onset'},
                    {'label': 'Expected Stimulus', 'value': 'Expected Stimulus'},
                    {'label': 'Servo In', 'value': 'Servo In'},
                    {'label': 'Choice', 'value': 'Choice'},
                    {'label': 'Outcome', 'value': 'Outcome'},
                    {'label': 'Trial End', 'value': 'Trial End'},
                ],
                value='Trial Start',
                clearable=False,
                className='mb-3'
            )
        ], md=4, className='p-2'),
        
        dbc.Col([
            html.Label('Outcome:', className='fw-bold mb-2'),
            dcc.Checklist(
                id='outcome-selector',
                options=[
                    {'label': ' Rewarded', 'value': 'rewarded'},
                    {'label': ' Punished', 'value': 'punished'}
                ],
                value=['rewarded', 'punished'],
                inline=True,
                className='mb-3'
            )
        ], md=4, className='p-2'),
        
        dbc.Col([
            html.Label('Plot Mode:', className='fw-bold mb-2'),
            dcc.RadioItems(
                id='plot-mode',
                options=[
                    {'label': ' Separate Subplots', 'value': 'separate'},
                    {'label': ' Superimposed', 'value': 'superimposed'}
                ],
                value='separate',
                inline=True,
                className='mb-3'
            )
        ], md=4, className='p-2'),
    ], className='mb-4 p-3 bg-light rounded shadow-sm'),
    
    dbc.Row([
        dbc.Col([
            html.Label('Select Conditions:', className='fw-bold mb-2'),
            dcc.Checklist(
                id='condition-selector',
                options=[
                    {'label': ' Short trials', 'value': 'Short trials'},
                    {'label': ' Long trials', 'value': 'Long trials'},
                    {'label': ' Short standard trials', 'value': 'Short standard trials'},
                    {'label': ' Short rare trials', 'value': 'Short rare trials'},
                    {'label': ' Long standard trials', 'value': 'Long standard trials'},
                    {'label': ' Long rare trials', 'value': 'Long rare trials'},
                    {'label': ' Early standard short trials', 'value': 'Early standard short trials'},
                    {'label': ' Late standard short trials', 'value': 'Late standard short trials'},
                    {'label': ' Early rare short trials', 'value': 'Early rare short trials'},
                    {'label': ' Late rare short trials', 'value': 'Late rare short trials'},
                    {'label': ' Early standard long trials', 'value': 'Early standard long trials'},
                    {'label': ' Late standard long trials', 'value': 'Late standard long trials'},
                    {'label': ' Early rare long trials', 'value': 'Early rare long trials'},
                    {'label': ' Late rare long trials', 'value': 'Late rare long trials'},
                ],
                value=['Short trials'],
                inline=False,
                className='mb-3'
            )
        ], md=12, className='p-2')
    ], className='mb-4 p-3 bg-light rounded shadow-sm'),
    
    dbc.Row([
        dbc.Col([
            dcc.Graph(id='neural-plot', style={'height': '80vh'})
        ], width=12)
    ]),
    
    dcc.Loading(
        id='loading',
        type='circle',
        children=[html.Div(id='loading-output')]
    )
], fluid=True, className='py-4 bg-secondary-subtle')

# Callback to update plot
@app.callback(
    Output('neural-plot', 'figure'),
    Output('loading-output', 'children'),
    Input('alignment-selector', 'value'),
    Input('condition-selector', 'value'),
    Input('outcome-selector', 'value'),
    Input('plot-mode', 'value')
)
def update_plot(alignment, conditions, outcomes, mode):
    """Update plot when alignment, conditions, outcomes, or mode change."""
    if not conditions or not outcomes:
        fig = go.Figure()
        fig.add_annotation(
            text='Please select at least one condition and one outcome.',
            xref='paper', yref='paper',
            x=0.5, y=0.5,
            showarrow=False,
            font=dict(size=20)
        )
        return fig, ''
    
    try:
        fig = data_storage.get_figure(alignment, conditions, outcomes, mode)
        return fig, ''
    except Exception as e:
        fig = go.Figure()
        fig.add_annotation(
            text=f'Error generating plot: {str(e)}',
            xref='paper', yref='paper',
            x=0.5, y=0.5,
            showarrow=False,
            font=dict(size=20)
        )
        return fig, ''


# Run the app
if __name__ == '__main__':
    print("Computing data for all alignments... This may take a minute...")
    try:
        # Replace neural_trials with _neural_trials (list of sessions)
        data_storage.compute_data(list_neural_trials, l_frames=60, r_frames=120)
        print("Data ready!")
    except NameError:
        print("Warning: _neural_trials not defined. Please load data before running compute_data.")
    
    print("Starting app...")
    app.run(debug=True)

Checking DataStorage methods: ['compute_data', 'data_cache', 'get_figure']
Computing data for all alignments... This may take a minute...


100%|██████████| 500/500 [00:00<00:00, 5494.44it/s]


Data ready!
Starting app...


C:\Users\ROG STRIX\AppData\Local\Temp\ipykernel_10988\3153907169.py:42: RuntimeWarning:

Mean of empty slice

C:\Users\ROG STRIX\AppData\Local\Temp\ipykernel_10988\3153907169.py:43: SmallSampleWarning:

All axis-slices of one or more sample arguments are too small; all elements of returned arrays will be NaN. See documentation for sample size requirements.

C:\Users\ROG STRIX\AppData\Local\Temp\ipykernel_10988\3153907169.py:42: RuntimeWarning:

Mean of empty slice

C:\Users\ROG STRIX\AppData\Local\Temp\ipykernel_10988\3153907169.py:43: SmallSampleWarning:

All axis-slices of one or more sample arguments are too small; all elements of returned arrays will be NaN. See documentation for sample size requirements.

C:\Users\ROG STRIX\AppData\Local\Temp\ipykernel_10988\3153907169.py:42: RuntimeWarning:

Mean of empty slice

C:\Users\ROG STRIX\AppData\Local\Temp\ipykernel_10988\3153907169.py:43: SmallSampleWarning:

All axis-slices of one or more sample arguments are too small; all elements o

# Raster plot

In [14]:
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.colors as mcolors
# import rastermap as rm

# def plot_rastermap_heatmap(neu_seq, neu_time, ax=None, ax_cb=None, 
#                            norm_mode="minmax", max_pixel=258):
#     """
#     Plot a heatmap of neural activity sorted with Rastermap.

#     Parameters
#     ----------
#     neu_seq : array (n_neurons x n_time)
#         Neuronal activity (e.g. dF/F).
#     neu_time : array (n_time,)
#         Timepoints corresponding to columns of neu_seq.
#     ax : matplotlib axis
#         Axis to plot heatmap on.
#     ax_cb : matplotlib axis
#         Axis for colorbar.
#     norm_mode : str
#         One of ['minmax', 'none'] for scaling across neurons.
#     max_pixel : int
#         Max number of rows to show (binning if > max_pixel).
#     """
#     if ax is None:
#         fig, ax = plt.subplots(figsize=(6,4))
#         ax_cb = fig.add_axes([0.92, 0.15, 0.02, 0.7])

#     # 1. Drop NaN-only rows
#     valid_idx = np.where(~np.all(np.isnan(neu_seq), axis=1))[0]
#     data = neu_seq[valid_idx, :].copy()

#     # 2. Smooth & sort with Rastermap
#     model = rm.Rastermap(n_clusters=3, locality=1)
#     model.fit(data)
#     sorted_idx = model.isort
#     data = data[sorted_idx, :]

#     # 3. Optional binning to limit pixels
#     nbin = max(1, data.shape[0] // max_pixel)
#     data = rm.utils.bin1d(data, bin_size=nbin, axis=0)

#     # 4. Normalization
#     if norm_mode == "minmax":
#         data = (data - np.nanmin(data, axis=1, keepdims=True)) / \
#                (np.nanmax(data, axis=1, keepdims=True) - np.nanmin(data, axis=1, keepdims=True) + 1e-9)
#         vmin, vmax = 0, 1
#     elif norm_mode == "none":
#         vmin, vmax = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
#     else:
#         raise ValueError("norm_mode must be 'minmax' or 'none'")

#     # 5. Plot heatmap
#     im = ax.imshow(data, aspect="auto", interpolation="nearest",
#                    extent=[neu_time[0], neu_time[-1], 1, data.shape[0]],
#                    cmap="hot", vmin=vmin, vmax=vmax)

#     ax.set_ylabel("Neuron (sorted)")
#     ax.set_xlabel("Time (s)")

#     # 6. Colorbar
#     if ax_cb is not None:
#         cbar = plt.colorbar(im, cax=ax_cb)
#         cbar.set_label("dF/F")

#     return ax, ax_cb

# fig, (ax, ax_cb) = plt.subplots(1, 2, gridspec_kw={'width_ratios':[20,1]}, figsize=(8,6))
# # Call get_perception_response once per column
# l_frames, r_frames = 60, 120
# indices = 0
# target_state = 'time_trial_start'
# neu_seq, neu_time, stim_seq, stim_value, stim_time, led_value, trial_type, block_type, isi, decision, outcome = get_perception_response(
#     neural_trials, target_state, l_frames, r_frames, indices=indices
# )
# neu_seq_mean = np.nanmean(neu_seq, axis=0)
# plot_rastermap_heatmap(neu_seq_mean, neu_time, ax=ax, ax_cb=ax_cb, norm_mode="minmax")
# plt.show()
